# Imports

In [1]:
from nb_utils import set_root
PROJECT_DIR = set_root(2)

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import plotly.graph_objects as go

# Parameters

In [3]:
path_data = PROJECT_DIR / 'data'
path_primary = path_data / "03_primary"

file_path_data = path_primary / "horm_tracker.parquet"
tracker_columns = [
    "tracker_id", "class_id", "x_min", "y_min", "x_max", "y_max", "x_center", "y_center"
]

# Read

In [ ]:
data = pd.read_parquet(file_path_data)
data.head()

In [5]:
data["ID"] = data["ID"].astype(int)

In [6]:
data_tracker = data[["ID"] + tracker_columns].copy()
data_horm = data.drop(tracker_columns, axis=1).drop_duplicates().copy()

# PCA

## 1) Hormonios

In [7]:
pca = PCA(n_components=2)
X_pca = pd.DataFrame(
    pca.fit_transform(data_horm.drop("ID", axis=1)),
    columns=['PCA1', 'PCA2']
)

loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PCA1', 'PCA2'],
    index=data_horm.drop("ID", axis=1).columns
).sort_values(["PCA1", "PCA2"], ascending=False)


In [ ]:
# Encontra a variável com o valor máximo em cada coluna
max_pc1_variable = loadings.loc[loadings['PCA1'].idxmax()].name
max_pc2_variable = loadings.loc[loadings['PCA2'].idxmax()].name

max_pc1_variable, max_pc2_variable

In [ ]:
fig = go.Figure()

colors = {
    'Q1': 'blue',    # Quadrante 1 (x > 0, y > 0)
    'Q2': 'green',   # Quadrante 2 (x < 0, y > 0)
    'Q3': 'red',     # Quadrante 3 (x < 0, y < 0)
    'Q4': 'purple'   # Quadrante 4 (x > 0, y < 0)
}

# Função para determinar o quadrante
def get_quadrant(x, y):
    if x > 0 and y > 0:
        return "Q1"
    elif x < 0 and y > 0:
        return "Q2"
    elif x < 0 and y < 0:
        return "Q3"
    else:
        return "Q4"

# Adicionando os pontos ao gráfico
for idx, index in enumerate(data_horm["ID"]):
    x = X_pca.loc[idx, "PCA1"]
    y = X_pca.loc[idx, "PCA2"]
    quadrant = get_quadrant(x, y)
    color = colors[quadrant]
    label = f"Quadrante {quadrant[-1]}"
    
    fig.add_trace(go.Scatter(
        x=[x], y=[y],
        mode="markers",
        marker=dict(color=color, size=10),
        name=label,
        customdata=[index],  # Adiciona o ID ao customdata
        hovertemplate="ID: %{customdata}<br>PCA1: %{x}<br>PCA2: %{y}<extra></extra>",  # Formata o texto de hover
        legendgroup=quadrant,  # Agrupa os pontos por quadrante
        showlegend=(label not in [trace.name for trace in fig.data])  # Mostra a legenda apenas uma vez por quadrante
    ))

# Adicionando as linhas divisórias
fig.add_trace(go.Scatter(
    x=[min(X_pca["PCA1"]), max(X_pca["PCA1"])],
    y=[0, 0],
    mode="lines",
    line=dict(color="red", dash="dash"),
    showlegend=False
))
fig.add_trace(go.Scatter(
    x=[0, 0],
    y=[min(X_pca["PCA2"]), max(X_pca["PCA2"])],
    mode="lines",
    line=dict(color="red", dash="dash"),
    showlegend=False
))

# Atualizando o layout
fig.update_layout(
    title="Scatter Plot do PCA de cada individuo",
    xaxis_title="PCA1",
    yaxis_title="PCA2",
    showlegend=True,
    height=900,  # Aumenta a altura do gráfico
    width=1500,   # Largura do gráfico
    legend=dict(itemsizing="constant")
)

fig.show()

In [12]:
data_horm.loc[((X_pca["PCA1"] > 0) & (X_pca["PCA2"] > 0)).values, "quadrante"] = "Q1"
data_horm.loc[((X_pca["PCA1"] < 0) & (X_pca["PCA2"] > 0)).values, "quadrante"] = "Q2"
data_horm.loc[((X_pca["PCA1"] < 0) & (X_pca["PCA2"] < 0)).values, "quadrante"] = "Q3"
data_horm.loc[((X_pca["PCA1"] > 0) & (X_pca["PCA2"] < 0)).values, "quadrante"] = "Q4"

In [ ]:
# selecionando features mais importantes e comparando quadrantes (utilizando leacao com o pca 2)
list_feats_impo = list(loadings["PCA2"].sort_values().index)
data_horm.drop("ID", axis=1).groupby(["quadrante"]).mean().loc[:,list_feats_impo[:5] + list_feats_impo[(-5):] ]

In [14]:
# data_horm["classe_test"] = 0
# data_horm.loc[data_horm["Age"] < 30, "classe_test"] = 0
# data_horm.loc[(data_horm["Age"] >= 30) & (data_horm["Age"] < 40), "classe_test"] = 1
# data_horm.loc[(data_horm["Age"] >= 40) & (data_horm["Age"] < 50), "classe_test"] = 2
# data_horm.loc[(data_horm["Age"] >= 50), "classe_test"] = 3
# data_horm[["classe_test", "Total sperm count", "Seminal AMH"]].groupby("classe_test").mean().join(
#     data_horm[["classe_test", "Total sperm count"]].groupby("classe_test").count().rename(columns={"Total sperm count": "linhas"}),
#     how="inner"
# )